# Lab 02 — From the One-Time Pad to Stream Ciphers

**IT4010E — Introduction to Cryptography and Security, SoICT, HUST**

**Nhóm thực hiện:**
- Nguyễn Hoàng Khôi — 202416710
- Nguyễn Quý Dương — 202416683
- Lê Trọng Đạt — 202416672

## Setup

Messages used throughout the lab, and the `strxor` helper shared by several tasks.

In [82]:
import os

M1 = b"Send the final report to the dean before Friday noon."
M2 = b"The exam for the course starts at eight in room D9."
M3 = b"Lunch will be served in the main hall at half past twelve."
MSGS = [M1, M2, M3]


def strxor(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(a, b))

## Task 1 — One-time pad

Port the Python 2 OTP snippet to Python 3 (`strxor`, `decrypt`), encrypt/decrypt M1, M2, M3 with a 1024-byte key, save the ciphertexts, then show what happens with a message longer than the key and how to refuse it.

In [83]:
def random_bytes(size=16):
    # os.urandom() reads from the same OS CSPRNG source as /dev/urandom on
    # Linux, and also works on Windows, where /dev/urandom does not exist.
    return os.urandom(size)


def encrypt(key: bytes, msg: bytes) -> bytes:
    c = strxor(key, msg)
    print(c.hex())
    return c


def encrypt_checked(key: bytes, msg: bytes) -> bytes:
    if len(msg) > len(key):
        raise ValueError(
            f"message ({len(msg)} bytes) longer than key ({len(key)} bytes)"
        )
    return encrypt(key, msg)


def decrypt(key: bytes, c: bytes) -> bytes:
    return strxor(key, c)

In [84]:
key = random_bytes(1024)
print("key =", key.hex())

print("\n--- Encrypting M1, M2, M3 ---")
ciphertexts = [encrypt(key, msg) for msg in MSGS]

with open("ciphertexts.txt", "w") as f:
    for c in ciphertexts:
        f.write(c.hex() + "\n")
print("\nSaved ciphertexts to ciphertexts.txt")

print("\n--- Decrypting ---")
for c in ciphertexts:
    p = decrypt(key, c)
    print(p.decode())

key = 5823d2e344a29ff649cb1746472ac4264feb90f2049b4ef0ddceaf75101098b767488dd537714a379ee49f9aca9b9287fc6de294b849341ddc3093bf28eb8c2717dacdf0be714dab3a7555afb0c97c9ba6872cdd9d1d33b81fff2440669546e7354ce68f26a299dd754e12eb0ff5c2c3197c5c28292a6bbd80c000ab49dc1b847fdeecf883db9020b99f221d45f461d4d59becb57c807b7864e05c78dee982bb6db41d1b32c0d9f66917ae70006a5c4bfa42a9302031685240d434608479b719d199fd8f9a8a07cc159bbd06178769d9adf714f8338b8ee13e307e33b3366cccda810f05ece3c3ec6c7e8171b9a5ba791c28c5924cf3b01c76a31e721c2ef38efce536ac76f36889028e7304282b126b13ffa01b70be706c6d4edb91e16e56d7dbf61a1cbbe57b614d175615776159713f8cc1b839e22d70da029ea5e2ede507b75f7b4992a054640f5669ae59990b2038118165a4c0ab0549be3b05529a612afe3d6f8e249b875e0bba6864faa1fb0f489b51fedc6d5f5dc4a77a70fd218d1bf49bf499e31540dd07f34536aac14edf4a2eae1038f45bdf5173e61519f6236ff16adfbfa788a28929eb247e00afced5b0c54a10a08b3a79a31da24662aecf9c1a14563b6c28655e335dce1a5f20aed4007c52561a7bd9903cd8a39022cc758db8603e55ff4e8110cafd84d0c1c788b944

In [85]:
print("--- Encrypting a 1500-byte message with a 1024-byte key ---")
long_msg = ((M1 + b" " + M2 + b" " + M3 + b" ") * 10)[:1500]
print("len(msg) =", len(long_msg))
c_long = encrypt(key, long_msg)
print("len(ciphertext) =", len(c_long))

p_long = decrypt(key, c_long)
print("decrypted back:", p_long.decode())
print("=> only the first", len(p_long), "bytes of the message survived,",
      "the rest was silently dropped by strxor/zip.")

print("\n--- Same message, but with encrypt_checked (refuses a message longer than the key) ---")
try:
    encrypt_checked(key, long_msg)
except ValueError as e:
    print("Rejected:", e)

--- Encrypting a 1500-byte message with a 1024-byte key ---
len(msg) = 1500
0b46bc8764d6f79369ad7e282646e4542a9bff8070bb3a9ffdbac7103074fdd60968efb0511e3852bea2edf3aefaeba792028dfa96696075b910f6c74986ac4178a8ed84d6146dc8550027dcd5e90fefc7f558aebd7c47987a96432812b52f89153e89e04b82dde45b6e5e9e6196aae36e15304409480e9df3a572dd2cb83bed11fe9890e6fbfd41d0f1027524980df4b4efccdd1dec1d5814812f0cfe9df5de01c278351293bc980d37da18654a3a229423c5105254183d32a01414eb59c371b4b999eafbe427ae70fdd27472a72fabc493758113e5e18e501e5e67db534ca9a2e062258a8cb1cc1816e451dacacf0b6f4de5e13892c26805837f063c4b9ae9949116c518d31ae66de35340110532276691c37350c91900016eb9f4c11d33a5ad937e3cd28b5b15257276781608375157edadd419835950b263f2c3c29d8474c37f0f3ef7cc220121763acb37fd2b545074a103cdaeca6969cc5e753de8150a8a524ffa4cfea73a6edb064498c49d603afe71b8ae043b3cbd87141f924fa33ba0f391b9866d21b027952a448ab526ba6a4dc1654a873eff220787676d85030e854abad6c0e0d6a94085040c6fc0a3f5f4fc6430ecfe541acb3dd52f0ec2effe7f34255e1e5e003a1334a03a2b48

**Giải thích:**

- Chạy thẳng đoạn code Python 2 gốc trong Python 3 sẽ báo `SyntaxError: Missing parentheses in call to 'print'`, vì `print` trong Python 2 là statement còn Python 3 yêu cầu gọi hàm `print(...)`. Dù sửa cú pháp này, `c.encode('hex')` vẫn lỗi vì `str.encode()` trong Python 3 chỉ nhận text encoding (như `'utf-8'`), codec `'hex'` (bytes→bytes) không còn dùng qua `.encode()` được nữa (phải dùng `c.hex()` nếu `c` là `bytes`). Ngoài ra `open("/dev/urandom")` không mở ở chế độ `'rb'` nên trả về `str`, XOR với `bytes` sẽ báo `TypeError`; và trên Windows, `/dev/urandom` không tồn tại nên sẽ báo `FileNotFoundError` ngay từ đầu — đó là lý do dùng `os.urandom()` thay thế.
- Với message 1500 byte: vì `strxor` dùng `zip(key, msg)`, phép zip dừng ở chuỗi ngắn hơn (key chỉ 1024 byte), nên ciphertext chỉ dài 1024 byte — 476 byte cuối của message bị **âm thầm cắt bỏ** chứ không báo lỗi. Đây là lý do cần sửa `encrypt` (bản `encrypt_checked`) để chủ động từ chối message dài hơn key thay vì để mất dữ liệu trong im lặng.

## Task 2 — Reusing the pad

Dùng lại `ciphertexts.txt` từ Task 1 (không dùng key): tính $c_1 \oplus c_2$, dò từ `" the "` bằng crib dragging, và giả mạo một key để $c_1$ giải mã ra một thông điệp tuỳ ý.

In [86]:
def is_lower_or_space(data: bytes) -> bool:
    return all((97 <= b <= 122) or b == 32 for b in data)


def load_ciphertexts(path="ciphertexts.txt"):
    with open(path) as f:
        lines = [line.strip() for line in f if line.strip()]
    return [bytes.fromhex(line) for line in lines]


c1, c2, c3 = load_ciphertexts()

In [87]:
print("--- Step 1: c1 xor c2 ---")
x12 = strxor(c1, c2)
print("c1 xor c2 =", x12.hex())

expected = strxor(M1, M2)
print("M1 xor M2 =", expected.hex())
print("equal?", x12 == expected)

--- Step 1: c1 xor c2 ---
c1 xor c2 = 070d0b44450c09080000061c41184817451300070653114f53000917541745001a00070c010706454928521b0b0e14002a5641
M1 xor M2 = 070d0b44450c09080000061c41184817451300070653114f53000917541745001a00070c010706454928521b0b0e14002a5641
equal? True


In [88]:
print("--- Step 2: crib dragging with \" the \" ---")
crib = b" the "
for i in range(len(x12) - len(crib) + 1):
    candidate = strxor(x12[i:i + len(crib)], crib)
    if is_lower_or_space(candidate):
        print(f"position {i:2d}: {candidate.decode()!r}")

--- Step 2: crib dragging with " the " ---
position  4: 'exam '
position  8: ' tnya'
position 12: 'al re'
position 24: 'start'


In [89]:
print("--- Step 3: forge a key for c1 ---")
target = b"Nothing to see here."
if len(target) < len(c1):
    target_padded = target + b" " * (len(c1) - len(target))
else:
    target_padded = target[:len(c1)]

k_prime = strxor(c1, target_padded)
print("k' =", k_prime.hex())

check = decrypt(k_prime, c1)
print("decrypt(k', c1) =", check.decode())
print("matches target?", check == target_padded)

--- Step 3: forge a key for c1 ---
k' = 4529c8ef0db890b31dc25e5b4323c43c4fe99aae509b1abfdd9ae7301054ddf62948cf90713e18729e82cdd38edacb87b222addab6
decrypt(k', c1) = Nothing to see here.                                 
matches target? True


**Giải thích:**

- **Bước 1:** $c_1 = k \oplus M_1$ và $c_2 = k \oplus M_2$ (cùng key $k$ vì pad bị dùng lại), nên $c_1 \oplus c_2 = k \oplus M_1 \oplus k \oplus M_2 = M_1 \oplus M_2$ — key $k$ tự triệt tiêu do $k \oplus k = 0$. Đây chính là lỗi chí mạng của việc dùng lại OTP: attacker không cần biết key vẫn tính được XOR của hai plaintext.
- **Bước 2:** Kết quả chạy cho 4 vị trí: `4: 'exam '`, `8: ' tnya'`, `12: 'al re'`, `24: 'start'`. Đối chiếu với M1 = `"Send the final report to the dean before Friday noon."` và M2 = `"The exam for the course starts at eight in room D9."`:
  - Vị trí 4: M1[4:9] thực sự là `" the "` (trong "Send **the** final...") → XOR crib vào lộ ra đúng M2[4:9] = `"exam "` — một từ thật của M2.
  - Vị trí 24: M1[24:29] cũng là `" the "` (trong "...report to **the** dean...") → lộ ra M2[24:29] = `"start"` — đúng đoạn đầu của từ "starts" trong M2.
  - Vị trí 12: lần này M2[12:17] mới là `" the "` (trong "...for **the** course...") → lộ ra M1[12:17] = `"al re"` — đúng đoạn cuối "...fin**al re**port..." của M1.
  - Vị trí 8 (`' tnya'`) là một **false positive**: không có `" the "` thật ở M1 hay M2 tại vị trí này, kết quả XOR chỉ *tình cờ* rơi vào toàn chữ thường/space chứ không phải từ có nghĩa — minh hoạ vì sao crib dragging cần con người (hoặc dictionary check) lọc lại các gợi ý.
  - Tóm lại: những vị trí cho ra **từ thật, có nghĩa** ứng với nơi crib `" the "` khớp đúng plaintext của message kia; những vị trí cho ra chuỗi không phải từ tiếng Anh (như `' tnya'`) là trùng hợp ngẫu nhiên của XOR, không mang thông tin thật.
- **Bước 3:** Vì OTP không có cơ chế xác thực (MAC/tag), attacker chỉ cần biết $c_1$ là đủ để tính ra một key $k'$ làm cho $c_1$ "giải mã" thành bất kỳ thông điệp nào có cùng độ dài — mà không hề biết key thật (ở đây $k' \oplus c_1 = $ "Nothing to see here." đúng như kết quả chạy). Điều này cho thấy OTP hoàn toàn không có tính *toàn vẹn/không thể chối bỏ*: không thể dùng ciphertext một mình để chứng minh nội dung gốc, vì bất kỳ ai cũng có thể "giải thích" nó theo một plaintext khác bằng cách chọn key phù hợp.

## Task 3 — A pseudorandom pad from NaCl

Dùng `SecretBox` (XSalsa20) của PyNaCl để "kéo dài" một key 32 byte thành pad giả ngẫu nhiên $G(k, \text{nonce}, n)$, viết `encrypt`/`decrypt`, rồi kiểm tra hậu quả của việc dùng lại nonce (giống lỗi Task 2).

In [90]:
import nacl.secret
import nacl.utils


def G(key: bytes, nonce: bytes, n: int) -> bytes:
    box = nacl.secret.SecretBox(key)
    # box.encrypt(data, nonce).ciphertext = 16-byte tag || (data xor stream)
    # with data = n zero bytes, this is tag || stream, so [16:] is the pad.
    return box.encrypt(bytes(n), nonce).ciphertext[16:]


def encrypt(key: bytes, msg: bytes) -> bytes:
    nonce = nacl.utils.random(24)
    pad = G(key, nonce, len(msg))
    return nonce + strxor(msg, pad)


def decrypt(key: bytes, c: bytes) -> bytes:
    nonce, body = c[:24], c[24:]
    pad = G(key, nonce, len(body))
    return strxor(body, pad)

In [91]:
key = nacl.utils.random(32)
nonce = nacl.utils.random(24)
print("key   =", key.hex())
print("nonce =", nonce.hex())
print("G(k, nonce, 64) =", G(key, nonce, 64).hex())

key   = 5d32a898cc9b473a563f72fe14192b902974d9fe5480de4aac34f22170949a56
nonce = e2031e7aff532b7156253676648e32a073fbd9c2a9c36208
G(k, nonce, 64) = 45d67f541cb4f0bca3cb5422f76e7eb0da47ab309d6676b16cb7eafaeefa50ade997f652517da6286564f495d4c60705af6be14b3776d028e177eb5872722282


In [92]:
print("--- Encrypting M1 twice (fresh random nonce each time) ---")
c1a = encrypt(key, M1)
c1b = encrypt(key, M1)
print("c1a =", c1a.hex())
print("c1b =", c1b.hex())
print("same ciphertext?", c1a == c1b)

p1a = decrypt(key, c1a)
print("decrypt(c1a) =", p1a.decode())

--- Encrypting M1 twice (fresh random nonce each time) ---
c1a = 3d861c1245f7f0bb35c1a08a86920287f5e68786b345e2f8ca6da4b678d8d6b8c3efabf8a5a508e1281b4cb33754c834cec5a3ef90be0bf6e083711dbab426c4a015c32ae6173772d6eb0199b2
c1b = d0d734c23d2566b802155c8e78edcdbf10e3dcb3bb8e5ca5c198508faa720b8cb0983ae98d53f7dbf11ec214a2987e60d7d3e2414255f9d3af4d63bbbf6f857fcb477626c2da5e7749bfa19018
same ciphertext? False
decrypt(c1a) = Send the final report to the dean before Friday noon.


In [93]:
print("--- Encrypting M1 and M2 with the SAME nonce ---")
reused_nonce = nacl.utils.random(24)
padA = G(key, reused_nonce, len(M1))
padB = G(key, reused_nonce, len(M2))
cA = reused_nonce + strxor(M1, padA)
cB = reused_nonce + strxor(M2, padB)

bodyA, bodyB = cA[24:], cB[24:]
x_bodies = strxor(bodyA, bodyB)
expected = strxor(M1, M2)
print("bodyA xor bodyB =", x_bodies.hex())
print("M1 xor M2       =", expected.hex())
print("equal?", x_bodies == expected)

--- Encrypting M1 and M2 with the SAME nonce ---
bodyA xor bodyB = 070d0b44450c09080000061c41184817451300070653114f53000917541745001a00070c010706454928521b0b0e14002a5641
M1 xor M2       = 070d0b44450c09080000061c41184817451300070653114f53000917541745001a00070c010706454928521b0b0e14002a5641
equal? True


**Giải thích:**

- Hai lần mã hoá M1 cho ra ciphertext khác nhau (`same ciphertext? False`) vì mỗi lần gọi `encrypt` đều tự sinh một **nonce ngẫu nhiên mới** (`nacl.utils.random(24)`), nên $G(k, \text{nonce})$ — pad giả ngẫu nhiên — cũng khác nhau dù dùng chung key $k$. Đây chính là *mã hoá xác suất* (probabilistic encryption): cùng plaintext, cùng key, nhưng ciphertext luôn đổi, giúp không lộ thông tin qua việc so sánh ciphertext.
- Khi dùng lại nonce cho hai message khác nhau, $G(k, \text{nonce}, \cdot)$ sinh ra **đúng cùng một pad**, nên pad tự triệt tiêu khi XOR hai ciphertext — kết quả là `bodyA xor bodyB == M1 xor M2` (`True`), giống hệt lỗi tái sử dụng OTP ở Task 2. Điều này cho thấy nonce trong stream cipher đóng đúng vai trò của "key dùng một lần" trong OTP: nếu nonce (cùng key) bị lặp lại, toàn bộ ưu điểm bảo mật của scheme sụp đổ y hệt như OTP bị tái sử dụng.

## Task 4 — Encrypting a book

Mã hoá `book.pdf` thành `book.enc` với một key 32 byte duy nhất, đọc file theo từng chunk 1 MiB, nonce = `prefix || i`. Giải mã lại và so sánh SHA-256; sau đó thử lỗi "quên tăng counter" (mọi chunk dùng `prefix || 0`).

In [94]:
import hashlib
import time

CHUNK_SIZE = 1024 * 1024  # 1 MiB


def int_xor(a: bytes, b: bytes) -> bytes:
    # XOR-ing as big integers is roughly 10x faster than a byte-by-byte
    # loop for large buffers (per the lab's tip).
    n = len(a)
    ai = int.from_bytes(a, "little")
    bi = int.from_bytes(b, "little")
    return (ai ^ bi).to_bytes(n, "little")


def encrypt_book(in_path, out_path, key, prefix=None, forget_counter=False):
    if prefix is None:
        prefix = nacl.utils.random(16)
    with open(in_path, "rb") as fin, open(out_path, "wb") as fout:
        fout.write(prefix)
        i = 0
        while True:
            chunk = fin.read(CHUNK_SIZE)
            if not chunk:
                break
            counter = 0 if forget_counter else i
            nonce = prefix + counter.to_bytes(8, "big")
            pad = G(key, nonce, len(chunk))
            fout.write(int_xor(chunk, pad))
            i += 1
    return prefix


def decrypt_book(in_path, out_path, key, forget_counter=False):
    with open(in_path, "rb") as fin, open(out_path, "wb") as fout:
        prefix = fin.read(16)
        i = 0
        while True:
            chunk = fin.read(CHUNK_SIZE)
            if not chunk:
                break
            counter = 0 if forget_counter else i
            nonce = prefix + counter.to_bytes(8, "big")
            pad = G(key, nonce, len(chunk))
            fout.write(int_xor(chunk, pad))
            i += 1


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(CHUNK_SIZE)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

In [95]:
book_key = nacl.utils.random(32)
print("book key =", book_key.hex())

print("\n--- Encrypting book.pdf -> book.enc (correct counter) ---")
t0 = time.perf_counter()
prefix = encrypt_book("book.pdf", "book.enc", book_key)
t1 = time.perf_counter()
print(f"encryption time: {t1 - t0:.3f} s")

size_pdf = os.path.getsize("book.pdf")
size_enc = os.path.getsize("book.enc")
print("size(book.pdf) =", size_pdf)
print("size(book.enc) =", size_enc)
print("size(book.enc) - size(book.pdf) =", size_enc - size_pdf)

with open("book.pdf", "rb") as f:
    first16_plain = f.read(16)
with open("book.enc", "rb") as f:
    f.read(16)  # skip the random prefix
    first16_cipher = f.read(16)
print("first 16 bytes of plaintext  =", first16_plain.hex())
print("first 16 bytes of ciphertext =", first16_cipher.hex())

book key = 39468e19f07df30f728b37a63a9a36091f8bc6e969d75d4c4e6fb335b3af0468

--- Encrypting book.pdf -> book.enc (correct counter) ---
encryption time: 0.124 s
size(book.pdf) = 9110736
size(book.enc) = 9110752
size(book.enc) - size(book.pdf) = 16
first 16 bytes of plaintext  = 255044462d312e350a25d0d4c5d80a38
first 16 bytes of ciphertext = 7a753b77adbf82e875f0118c57ac5585


In [96]:
print("SHA-256(book.pdf) =", sha256_file("book.pdf"))

decrypt_book("book.enc", "book.dec.pdf", book_key)
print("SHA-256(book.dec.pdf) =", sha256_file("book.dec.pdf"))
print("match?", sha256_file("book.pdf") == sha256_file("book.dec.pdf"))

SHA-256(book.pdf) = 67bc4507f7558c99f0ecc331651da224f0a73b2453fdd8ff5e67c8d4e56cc7dd
SHA-256(book.dec.pdf) = 67bc4507f7558c99f0ecc331651da224f0a73b2453fdd8ff5e67c8d4e56cc7dd
match? True


In [97]:
print("--- Encrypting again with the counter forgotten (every chunk uses prefix || 0) ---")
encrypt_book("book.pdf", "book.buggy.enc", book_key, forget_counter=True)


def read_two_chunks(path, skip_prefix):
    with open(path, "rb") as f:
        f.read(skip_prefix)
        c0 = f.read(CHUNK_SIZE)
        c1 = f.read(CHUNK_SIZE)
    return c0, c1


m0, m1 = read_two_chunks("book.pdf", skip_prefix=0)
c0_good, c1_good = read_two_chunks("book.enc", skip_prefix=16)
c0_bad, c1_bad = read_two_chunks("book.buggy.enc", skip_prefix=16)

print("\nCorrect-counter version:  c0 xor c1 == m0 xor m1 ?",
      int_xor(c0_good, c1_good) == int_xor(m0, m1))
print("Forgotten-counter version: c0 xor c1 == m0 xor m1 ?",
      int_xor(c0_bad, c1_bad) == int_xor(m0, m1))

--- Encrypting again with the counter forgotten (every chunk uses prefix || 0) ---

Correct-counter version:  c0 xor c1 == m0 xor m1 ? False
Forgotten-counter version: c0 xor c1 == m0 xor m1 ? True


**Giải thích:**

- **Key vs kích thước sách:** key chỉ 32 byte trong khi `book.pdf` nặng tới ~8.7 MB — chênh lệch hàng trăm nghìn lần. Đây chính là mục đích của stream cipher/PRG: không cần pad dài bằng message như OTP, chỉ cần "kéo giãn" một key rất ngắn thành keystream dài tuỳ ý bằng $G(k, \text{nonce})$.
- **Vì sao `book.enc` dài hơn `book.pdf` đúng 16 byte:** phần thân ciphertext (XOR với pad) có độ dài bằng hệt plaintext (không cộng thêm gì, vì ta chỉ lấy phần pad thô từ `G`, bỏ tag 16 byte của `SecretBox`). Overhead duy nhất trong cả file là **16 byte prefix** ghi một lần ở đầu file — counter của từng chunk không được lưu trong file mà được tính lại từ chỉ số chunk lúc giải mã, nên overhead không phụ thuộc vào kích thước sách hay số lượng chunk.
- **Kết quả "quên tăng counter":** ở bản đúng, mỗi chunk dùng nonce khác nhau (`prefix || i`) nên `c0 xor c1 == m0 xor m1` cho `False`. Ở bản lỗi, mọi chunk dùng cùng nonce `prefix || 0` → $G(k,\text{nonce},\cdot)$ sinh ra **cùng một keystream** cho mọi chunk, pad tự triệt tiêu khi XOR hai ciphertext chunk, nên `c0 xor c1 == m0 xor m1` cho `True` — đây chính xác là lỗi "two-time pad" ở Task 2/3 nhưng xảy ra *ngay trong cùng một file*, giữa các chunk với nhau, chứ không cần hai file khác nhau.
- **Vì sao giải mã bằng đúng code lỗi đó vẫn ra sách mở được:** vì `decrypt_book` dùng **chính công thức tính nonce/pad giống hệt lúc mã hoá** (dù công thức đó sai/lặp lại). Với bất kỳ pad nào (đúng hay sai), XOR luôn tự nghịch đảo: $c = m \oplus \text{pad} \Rightarrow c \oplus \text{pad} = m$. Lỗi quên tăng counter chỉ phá vỡ **tính bảo mật** (làm lộ quan hệ XOR giữa các chunk plaintext cho kẻ tấn công), chứ không phá vỡ **tính đúng đắn chức năng** khi cả hai phía (mã hoá và giải mã) đều nhất quán dùng cùng một (dù sai) cách sinh nonce.

## Task 5 — Changing an encrypted amount

Tấn công bit-flipping: sửa ciphertext của `"PAY BOB 0100 USD"` (mã hoá bằng cipher Task 3) mà không cần biết key, để nó giải mã thành `"PAY BOB 9900 USD"`. Lặp lại với `SecretBox` thật để thấy tag 16 byte chặn tấn công này.

In [98]:
print("--- Attack 1: bit-flip on our Task 3 cipher ---")
msg = b"PAY BOB 0100 USD"
c = encrypt(key, msg)
print("original ciphertext =", c.hex())
print("decrypts to          =", decrypt(key, c).decode())

target = b"PAY BOB 9900 USD"
delta = strxor(msg, target)  # nonzero only where the digits differ
print("delta (msg xor target) =", delta.hex())

c_forged = bytearray(c)
body = strxor(bytes(c_forged[24:]), delta)  # flip only the ciphertext body
c_forged[24:] = body
c_forged = bytes(c_forged)

print("forged ciphertext   =", c_forged.hex())
print("forged decrypts to  =", decrypt(key, c_forged).decode())

--- Attack 1: bit-flip on our Task 3 cipher ---
original ciphertext = 6ae32402d807f975833097910ad21f5f3aad7bdbd202f67926b38379432d42865d33287c716681f3
decrypts to          = PAY BOB 0100 USD
delta (msg xor target) = 00000000000000000908000000000000
forged ciphertext   = 6ae32402d807f975833097910ad21f5f3aad7bdbd202f67926b38379432d4286543b287c716681f3
forged decrypts to  = PAY BOB 9900 USD


In [99]:
print("--- Attack 2: same bit-flip on a real SecretBox ciphertext ---")
box = nacl.secret.SecretBox(key)
box_c = box.encrypt(msg)  # layout: nonce (24B) || tag (16B) || body
print("box ciphertext =", box_c.hex())

box_c_forged = bytearray(box_c)
body_offset = 24 + 16
body = strxor(bytes(box_c_forged[body_offset:]), delta)
box_c_forged[body_offset:] = body
box_c_forged = bytes(box_c_forged)
print("forged box ciphertext =", box_c_forged.hex())

try:
    result = box.decrypt(bytes(box_c_forged))
    print("box.decrypt succeeded:", result)
except Exception as e:
    print("box.decrypt raised:", type(e).__name__, "-", e)

--- Attack 2: same bit-flip on a real SecretBox ciphertext ---
box ciphertext = f517fa9c7e098b49f2d05e7c8f930dd3b97f0dc6cd086d313a8ec2ee2ea0fbc349dfb20f6891c3abab1dc0c397054ace59e1978c587ee364
forged box ciphertext = f517fa9c7e098b49f2d05e7c8f930dd3b97f0dc6cd086d313a8ec2ee2ea0fbc349dfb20f6891c3abab1dc0c397054ace50e9978c587ee364
box.decrypt raised: CryptoError - Decryption failed. Ciphertext failed verification


**Giải thích:**

- **Vì sao tấn công 1 thành công:** với stream cipher kiểu OTP/CTR, $c = m \oplus \text{pad}$. Nếu attacker biết (hoặc đoán được) plaintext gốc $m$ và muốn nó giải mã thành $m'$, chỉ cần tính $\delta = m \oplus m'$ rồi XOR $\delta$ vào đúng vị trí tương ứng trong $c$: $c' = c \oplus \delta = (m \oplus \text{pad}) \oplus (m \oplus m') = m' \oplus \text{pad}$, nên $c'$ giải mã đúng thành $m'$ — **hoàn toàn không cần biết key hay pad**. Đây gọi là tính **malleable** (dễ bị sửa) của mã hoá dạng XOR/CTR: sửa ciphertext một cách có kiểm soát sẽ tạo ra thay đổi có thể đoán trước trên plaintext, dù attacker không đọc được nội dung.
- **Tag 16 byte thêm gì:** `SecretBox` không chỉ mã hoá mà còn gắn kèm một **MAC (Message Authentication Code)** tính trên toàn bộ ciphertext bằng một sub-key riêng (Poly1305). Khi attacker sửa dù chỉ 1 bit trong body, MAC không còn khớp nữa (vì attacker không biết sub-key để tính lại MAC hợp lệ) → `box.decrypt` phát hiện sai lệch và **từ chối giải mã** (ném ngoại lệ `CryptoError`) thay vì âm thầm trả về plaintext bị sửa. Đây chính là lý do các hệ mã hiện đại luôn kết hợp mã hoá **+ xác thực** (authenticated encryption, AE) — mã hoá đơn thuần (kể cả OTP hoàn hảo về mặt bí mật) không đảm bảo tính toàn vẹn của dữ liệu.

## Task 6 — A shift-register generator

Cài đặt thanh ghi dịch phản hồi tuyến tính (LFSR): tính chu kỳ với vài bộ taps/seed, rồi dùng LFSR $n=32$, taps $\{0,10,30,31\}$ làm keystream để mã hoá một email mẫu.

In [78]:
class ShiftRegister:
    def __init__(self, n: int, taps, seed: int):
        if seed == 0:
            raise ValueError("seed must not be 0 (the all-zero row never changes)")
        self.n = n
        self.taps = taps
        self.mask = (1 << n) - 1
        self.row = seed & self.mask

    def step(self) -> int:
        out_bit = self.row & 1
        fb = 0
        for t in self.taps:
            fb ^= (self.row >> t) & 1
        self.row = (self.row >> 1) | (fb << (self.n - 1))
        return out_bit

    def bits(self, count: int):
        return [self.step() for _ in range(count)]

    def keystream(self, nbytes: int) -> bytes:
        out = bytearray(nbytes)
        for i in range(nbytes):
            b = 0
            for bit_pos in range(8):
                b |= self.step() << bit_pos
            out[i] = b
        return bytes(out)


def period(n: int, taps, seed: int) -> int:
    reg = ShiftRegister(n, taps, seed)
    start = reg.row
    steps = 0
    while True:
        reg.step()
        steps += 1
        if reg.row == start:
            return steps

In [79]:
print("--- Part 1: n=4, taps={0,1}, seed=1001 (9), first 16 output bits ---")
reg = ShiftRegister(4, {0, 1}, 0b1001)
print(reg.bits(16))

--- Part 1: n=4, taps={0,1}, seed=1001 (9), first 16 output bits ---
[1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1]


In [80]:
print("--- Part 2: periods ---")
print("n=4,  taps={0,1},       seed=0b1001 ->", period(4, {0, 1}, 0b1001))
print("n=16, taps={0,2,3,5},   seed=1      ->", period(16, {0, 2, 3, 5}, 1))
print("n=16, taps={0,8},       seed=1      ->", period(16, {0, 8}, 1))

--- Part 2: periods ---
n=4,  taps={0,1},       seed=0b1001 -> 15
n=16, taps={0,2,3,5},   seed=1      -> 65535
n=16, taps={0,8},       seed=1      -> 24


In [81]:
def random_nonzero_seed(n_bits: int) -> int:
    while True:
        s = int.from_bytes(os.urandom(n_bits // 8), "big")
        if s != 0:
            return s


print("--- Part 3: n=32, taps={0,10,30,31}, random seed ---")
message6 = (
    b"From: exam-office@example.edu\n"
    b"Subject: final exam\n"
    b"Room B1-401, 8:00 on Monday."
)

seed6 = random_nonzero_seed(32)
reg_msg = ShiftRegister(32, {0, 10, 30, 31}, seed6)
ks6 = reg_msg.keystream(len(message6))
cipher6 = strxor(message6, ks6)

print(f"seed = {seed6:08x}")
print("ciphertext =", cipher6.hex())

reg_sample = ShiftRegister(32, {0, 10, 30, 31}, seed6)
sample = reg_sample.keystream(64 * 1024)
ones = sum(bin(b).count("1") for b in sample)
total_bits = len(sample) * 8
print(f"fraction of 1-bits in 64 KiB = {ones / total_bits:.4f}")

--- Part 3: n=32, taps={0,10,30,31}, random seed ---
seed = bdcc4ae6
ciphertext = a038a3d0ab9e857d332c8da9a6f3ef84c4512cfa9b36887ef1d4f7587bf83a271fc9b73ae8a661affaf731381c8ab952acd50b9953ae5a4a05b291aa41fd7b790db1684f05c460b1198b9bec4d70
fraction of 1-bits in 64 KiB = 0.5005


**Giải thích:**

- **Vì sao seed không được là 0:** nếu `row = 0`, mọi bit tại các vị trí tap đều là 0, nên XOR phản hồi luôn ra 0 — dòng thanh ghi mãi mãi giữ nguyên `000...0` và output luôn toàn bit 0. Đây là một "điểm bất động" (fixed point) vô dụng: keystream hoàn toàn có thể đoán trước (toàn số 0), tương đương như không mã hoá gì cả.
- **Vì sao taps $\{0,8\}$ (với $n=16$) tạo pad tồi:** kết quả đo ở Part 2 cho thấy rất rõ sự khác biệt. Hai bộ taps $\{0,1\}$ (n=4) và $\{0,2,3,5\}$ (n=16) đều đạt đúng chu kỳ tối đa lý thuyết $2^n-1$ (lần lượt là 15 và 65535) — đây là các bộ taps ứng với một *đa thức hồi quy nguyên thuỷ* (primitive polynomial), cho thanh ghi đi qua tất cả $2^n-1$ trạng thái khác 0 trước khi lặp lại. Ngược lại, bộ taps $\{0,8\}$ chỉ cho chu kỳ **24** — ngắn hơn mức tối đa 65535 tới hơn 2700 lần. Lý do nằm ở việc tap thứ hai đặt đúng ở giữa thanh ghi ($8 = n/2$), khiến đa thức hồi quy tương ứng không phải là nguyên thuỷ (nó phân tích được thành các nhân tử bậc thấp hơn), nên quỹ đạo trạng thái rơi vào một chu kỳ con rất ngắn thay vì đi hết toàn bộ không gian trạng thái. Hệ quả thực tế: keystream **lặp lại chỉ sau 24 bit** — nếu dùng để mã hoá dữ liệu dài hơn thế, các đoạn ciphertext cách nhau bội số của 24 bit sẽ bị XOR với **đúng cùng một đoạn pad**, tạo ra chính xác lỗi "two-time pad" như ở Task 2 (kẻ tấn công tính được XOR của hai đoạn plaintext ở cùng pha, mà không cần biết seed). Vì vậy khi chọn taps cho LFSR, không thể chọn tuỳ ý mà phải chọn đúng bộ ứng với một đa thức nguyên thuỷ để đạt chu kỳ tối đa $2^n-1$.

## Task 7 — Breaking the shift register

Từ ciphertext cho trước và 8 byte đầu plaintext đã biết (`"From: ex"`), khôi phục seed và toàn bộ message (dòng cuối là flag). Bonus: khôi phục mà không cần biết taps, chỉ biết thanh ghi ≤ 32 bit (Berlekamp–Massey).

In [ ]:
CIPHER7 = bytes.fromhex(
    "3d5cae3331120fe78d2359b8998d846d5230dc78741f3880d0e36e3fb46597964"
    "a5841c015c4e29f25bbdcb0f946993cd2af3984176325a9688ab8ab6d07dfda"
    "f2f680eee88923ccc31738ce4c3c9962a5b92873b3064e64a69054def9c71331"
)
print("ciphertext length =", len(CIPHER7), "bytes")

KNOWN_PLAIN = b"From: ex"
KNOWN_LEN = len(KNOWN_PLAIN)

# XOR the known 8 plaintext bytes with the matching ciphertext bytes to
# recover the first 64 output bits of the register (bit 0 of byte 0 first,
# same packing as ShiftRegister.keystream).
known_ks_bytes = strxor(CIPHER7[:KNOWN_LEN], KNOWN_PLAIN)
known_bits = []
for byte in known_ks_bytes:
    for bit_pos in range(8):
        known_bits.append((byte >> bit_pos) & 1)
print("known output bits (64) =", known_bits)

In [ ]:
print("--- Recovering the seed (taps are known: {0,10,30,31}, n=32) ---")
# Key fact: output bit b_t (for t = 0..n-1) is exactly bit t of the seed --
# the register hasn't had time to feed any feedback bit back to the output
# yet, so the first n outputs are literally the seed read off LSB-first.
seed7 = 0
for j in range(32):
    seed7 |= known_bits[j] << j
print(f"recovered seed = {seed7:08x}")

# sanity check against all 64 known bits, using the (given) taps
reg_check = ShiftRegister(32, {0, 10, 30, 31}, seed7)
check_bytes = reg_check.keystream(KNOWN_LEN)
print("keystream check matches known bytes?", check_bytes == known_ks_bytes)

In [ ]:
print("--- Decrypting the full ciphertext ---")
reg_full = ShiftRegister(32, {0, 10, 30, 31}, seed7)
keystream_full = reg_full.keystream(len(CIPHER7))
message7 = strxor(CIPHER7, keystream_full)
print(message7.decode())

## Bonus — Recovering the message without knowing the taps

Chỉ dùng 64 bit keystream đã biết và giả thiết thanh ghi có tối đa 32 bit — không giả định taps $\{0,10,30,31\}$.

**Ý tưởng:** có thể chứng minh output $b_t$ của thanh ghi thoả một hệ thức truy hồi tuyến tính chuẩn $b_i = \bigoplus_{j=1}^{L} c_j\, b_{i-j}$ với $L=n$ (suy ra trực tiếp từ định nghĩa thanh ghi, xem phần giải thích bên dưới). Vì đây đúng là định nghĩa của một dãy LFSR chuẩn, thuật toán **Berlekamp–Massey** có thể tìm ra $L$ và các hệ số $c_j$ chỉ từ $2L$ bit liên tiếp của dãy — ở đây $2 \times 32 = 64$ bit, đúng bằng số bit ta có từ 8 byte plaintext đã biết.

In [ ]:
def berlekamp_massey(s):
    """Shortest LFSR (over GF(2)) generating the bit sequence s.
    Returns (L, C) with C[0] = 1 and s[i] = xor_{j=1}^{L} C[j] * s[i-j] for i >= L.
    """
    n = len(s)
    C = [1] + [0] * n
    B = [1] + [0] * n
    L, m = 0, 1
    for i in range(n):
        d = s[i]
        for j in range(1, L + 1):
            d ^= C[j] & s[i - j]
        if d == 0:
            m += 1
        elif 2 * L <= i:
            T = C.copy()
            for j in range(len(B)):
                if j + m < len(C):
                    C[j + m] ^= B[j]
            L = i + 1 - L
            B = T
            m = 1
        else:
            for j in range(len(B)):
                if j + m < len(C):
                    C[j + m] ^= B[j]
            m += 1
    return L, C[:L + 1]


L_found, C_found = berlekamp_massey(known_bits)
print("recovered register length L =", L_found)
print("recovered connection polynomial C =", C_found)

In [ ]:
print("--- Extending the sequence with the recovered recurrence ---")
total_bits_needed = len(CIPHER7) * 8
ext_bits = list(known_bits)
for i in range(len(ext_bits), total_bits_needed):
    val = 0
    for j in range(1, L_found + 1):
        val ^= C_found[j] & ext_bits[i - j]
    ext_bits.append(val)

# pack back into bytes, bit 0 (LSB) first -- same convention as ShiftRegister.keystream
ext_keystream = bytearray(len(CIPHER7))
for i in range(len(CIPHER7)):
    b = 0
    for bit_pos in range(8):
        b |= ext_bits[8 * i + bit_pos] << bit_pos
    ext_keystream[i] = b
ext_keystream = bytes(ext_keystream)

message7_bonus = strxor(CIPHER7, ext_keystream)
print(message7_bonus.decode())
print("\nmatches the taps-based decryption?", message7_bonus == message7)

**Giải thích:**

- **Vì sao 8 byte đã biết là đủ, dù output của Task 6.3 "trông ngẫu nhiên":** "trông ngẫu nhiên" chỉ có nghĩa là dãy bit vượt qua các kiểm tra thống kê đơn giản (như tỉ lệ bit 1 ≈ 0.5 đã đo ở Task 6.3). Nhưng về bản chất, mỗi bit output của thanh ghi dịch là một **hàm tuyến tính (XOR)** của các bit trước đó — có thể chứng minh trực tiếp từ định nghĩa: bit thứ $j$ của trạng thái tại thời điểm $t$ bằng $b_{t+j}$ (với $j<n$), từ đó suy ra $b_{t+n} = \bigoplus_{k\in\text{taps}} b_{t+k}$ với **mọi** $t$ — chính là một hệ thức truy hồi tuyến tính bậc $n$ chuẩn. Hệ quả:
  - Chỉ cần $n=32$ bit output liên tiếp là đọc ra được **chính xác seed** (không cần giải hệ phương trình gì cả — seed = 32 bit output đầu tiên, vì trong $n$ bước đầu chưa có bit feedback nào kịp quay về vị trí output).
  - Chỉ cần $2n=64$ bit là đủ để Berlekamp–Massey xác định trọn vẹn cả bậc $L$ lẫn các hệ số hồi quy (tương đương taps), theo đúng định lý Massey — không cần biết trước taps.
  - 8 byte = 64 bit khớp chính xác với cả hai ngưỡng trên. Việc "trông ngẫu nhiên" hoàn toàn không chống lại được tấn công đại số tuyến tính này — độ phức tạp phá vỡ chỉ là $O(n^2)$ hoặc tốt hơn, cực nhanh so với $2^n$ phép thử vét cạn mà kích thước không gian trạng thái gợi ý.
- **Thanh ghi 256-bit có an toàn không:** **Không.** Điểm yếu ở đây không nằm ở độ dài thanh ghi mà ở tính **tuyến tính** của phép cập nhật trạng thái. Với $n=256$, attacker chỉ cần nhiều hơn — $2n = 512$ bit ($64$ byte) plaintext đã biết thay vì $64$ bit — và Berlekamp–Massey vẫn chạy trong thời gian đa thức ($O(n^2)$, tức khoảng $256^2 \approx 65536$ bước, vẫn cực nhanh trên máy tính hiện đại) để khôi phục toàn bộ cấu trúc hồi quy rồi dự đoán mọi bit keystream còn lại. Đây chính là lý do các stream cipher hiện đại (như XSalsa20 dùng ở Task 3/4) **không bao giờ** dùng trực tiếp output của một LFSR thuần tuý làm keystream — chúng dùng các phép biến đổi **phi tuyến** (non-linear) phức tạp để việc lập hệ phương trình tuyến tính như Berlekamp–Massey không còn khả thi, bất kể kích thước trạng thái nội bộ lớn đến đâu.